# soundstream demo

this notebook clones the repository, installs the dependencies, downloads the trained soundstream checkpoint and reconstructs any audio file provided by URL

## 1. clone the repository

In [1]:
import os
from pathlib import Path

REPO_URL = "https://github.com/akhasanovv/soundstream-implementation.git"
REPO_DIR = Path("/soundstream-implementation")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"repository already exists at {REPO_DIR}")

os.chdir(REPO_DIR)
print("working directory:", Path.cwd())

repository already exists at /soundstream-implementation
working directory: /soundstream-implementation


## 2. install dependencies

In [2]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.2/78.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 133.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.

## 3. download the trained model

In [2]:
import requests

MODEL_CHECKPOINT_URL = "https://huggingface.co/akhasanovv/soundstream-implementation/resolve/main/model_best.pth"
CHECKPOINT_PATH = REPO_DIR / "saved" / "soundstream_model" / "model_best.pth"
CHUNK_SIZE = int(2**20)

def download_file(url: str, dst: Path):
    """download file from url and save it to dst

    Args:
        url (str): url of file
        dst (Path): destination of downloaded file

    Returns:
        dst (Path): path to downloaded file
    """
    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.exists() and dst.stat().st_size > 0:
        print(f"checkpoint already exists: {dst}")
        return dst

    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    with dst.open("wb") as file:
        for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
            if chunk:
                file.write(chunk)

    print(f"file from {url} downloaded to {dst}")
    return dst


download_file(MODEL_CHECKPOINT_URL, CHECKPOINT_PATH)

checkpoint already exists: /soundstream-implementation/saved/soundstream_model/model_best.pth


PosixPath('/soundstream-implementation/saved/soundstream_model/model_best.pth')

## 4. load soundstream model

In [ ]:
import torch
from src.model import SoundStreamModel

SAMPLE_RATE = 16000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_soundstream(path, device=DEVICE):
    model = SoundStreamModel(
        in_channels=32,
        out_channels=128,
        num_quantizers=8,
        num_embeddings=1024
    ).to(device)

    checkpoint = torch.load(path, map_location=device, weights_only=False)
    state_dict = checkpoint.get("state_dict", checkpoint) if isinstance(checkpoint, dict) else checkpoint
    model.load_state_dict(state_dict)
    model.eval()

    return model

model = load_soundstream(CHECKPOINT_PATH)
print(f"loaded soundstream model on {DEVICE}")

loaded soundstream model on cuda


## 5. reconstruct audio from URL

`reconstruct_audio` downloads an audio file, converts it to mono 16 kHz, runs it through the soundstream model, saves the reconstructed waveform, and displays both the original and reconstructed samples

In [4]:
import torchaudio
from IPython.display import Audio, display
import urllib.parse

def download_audio(url, dst_dir=Path("/content")) -> Path:
    suffix = Path(urllib.parse.urlparse(url).path).suffix or ".wav"
    dst = dst_dir / f"audio{suffix}"
    download_file(url, dst)
    return dst

def prepare_audio(audio_path, target_sr=SAMPLE_RATE):
    waveform, sr = torchaudio.load(audio_path)
    waveform = waveform.mean(dim=0, keepdim=True)

    if sr != target_sr:
        waveform = torchaudio.functional.resample(waveform, orig_freq=sr, new_freq=target_sr)

    waveform = waveform.clamp(-1.0, 1.0)
    return waveform, target_sr


def reconstruct_audio(audio_url, output_path="/content/reconstructed.wav"):
    """given audio_url, produces resontructed version of audio located in output_path

    Args:
        audio_url (str): _description_
        output_path (str, optional): _description_. Defaults to "/content/reconstructed.wav".

    Returns:
        _type_: _description_
    """
    input_path = download_audio(audio_url)
    waveform, sr = prepare_audio(input_path)

    with torch.inference_mode():
        audio = waveform.unsqueeze(0).to(DEVICE)
        reconstructed = model(audio)["reconstructed_audio"].squeeze(0).cpu().clamp(-1.0, 1.0)

    output_path = Path(output_path)
    torchaudio.save(output_path, reconstructed, sr)

    print("original audio")
    display(Audio(waveform.squeeze().numpy(), rate=sr))

    print("reconstructed audio")
    display(Audio(reconstructed.squeeze().numpy(), rate=sr))

    print(f"saved reconstructed audio to {output_path}")
    return reconstructed, sr, output_path


AUDIO_URL = "https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav" # sample url
reconstructed_audio, sample_rate, output_path = reconstruct_audio(AUDIO_URL)

file from https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav downloaded to /content/audio.wav
original audio


reconstructed audio


saved reconstructed audio to /content/reconstructed.wav
